# NeMo Magpie-TTS Primer

This notebook introduces modern text-to-speech with [Magpie-TTS](https://huggingface.co/nvidia/magpie_tts_multilingual_357m), a multilingual, codec-based speech model in NVIDIA NeMo.

You will learn how to:

- load the public 357M Magpie-TTS checkpoint;
- synthesize speech in multiple languages and baked voices;
- inspect the attached neural audio codec;
- encode waveforms into discrete audio tokens and decode them back to audio;
- measure basic generation quality and speed.

## 1. Setup

A CUDA GPU is strongly recommended. The first model load downloads the Magpie-TTS checkpoint and its configured audio codec from Hugging Face.

When running from a local Speech checkout, install the repository dependencies with the project installation guide. The cell below is convenient for a clean hosted notebook.

In [ ]:
# Install NeMo Speech with TTS dependencies in a clean notebook environment.
# Skip this cell when the current checkout is already installed.
BRANCH = "main"
!python -m pip install "nemo_toolkit[tts] @ git+https://github.com/NVIDIA-NeMo/Speech.git@$BRANCH"

In [ ]:
from pathlib import Path
import time

import IPython.display as ipd
import matplotlib.pyplot as plt
import soundfile as sf
import torch

from nemo.collections.tts.models import MagpieTTSModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)

print(f"Using {device}")

## 2. From text to codec tokens to audio

The older neural TTS stack often predicts a mel spectrogram and then uses a separate vocoder to turn that spectrogram into a waveform. Magpie-TTS uses a different representation:

1. **Text processing** normalizes and tokenizes the input for the selected language.
2. **Magpie-TTS** generates discrete audio tokens with monotonic alignment between text and speech.
3. **Neural audio codec** decodes those tokens directly into a waveform.

The codec is part of the model configuration, so the public checkpoint loads the matching codec automatically.

### Why discrete audio tokens?

A neural audio codec compresses a waveform into a short sequence of integer codes. Each time step contains one token from each codec codebook. The tokens preserve speech content, speaker identity, and acoustic detail while giving Magpie-TTS a compact vocabulary to generate.

For a tensor shaped `(batch, codebooks, frames)`:

- **frames** advance through time;
- **codebooks** describe complementary acoustic details at each frame;
- the codec decoder reconstructs the waveform from the complete token grid.

## 3. Load Magpie-TTS and its codec

The public checkpoint supports a variety of languages including: English, Spanish, German, French, Vietnamese, Italian, Mandarin Chinese, Hindi, and Japanese. It also contains five baked speaker embeddings.

In [ ]:
MODEL_NAME = "nvidia/magpie_tts_multilingual_357m"

model = MagpieTTSModel.from_pretrained(MODEL_NAME, map_location="cpu")
model = model.eval().to(device)

print(f"Model: {MODEL_NAME}")
print(f"Baked speakers: {model.num_baked_speakers}")
print(f"Waveform sample rate: {model.output_sample_rate} Hz")

In [ ]:
# Magpie-TTS owns the codec used for token generation and waveform decoding.
# Accessing _codec_model is useful for this educational inspection; normal
# synthesis should use model.do_tts(), which handles codec decoding internally.
codec = model._codec_model
codec_name = model.cfg.get("codecmodel_path")

print(f"Codec: {codec_name}")
print(f"Codec input sample rate: {codec.sample_rate} Hz")
print(f"Codec output sample rate: {codec.output_sample_rate} Hz")
print(f"Codebooks: {codec.num_codebooks}")
print(f"Codebook size: {codec.codebook_size}")
print(f"Samples per codec frame: {codec.samples_per_frame}")
print(f"Codec frames per second: {codec.sample_rate / codec.samples_per_frame:.2f}")

## 4. Synthesize speech

`do_tts()` is the recommended single-utterance API. It tokenizes the transcript, generates audio-codec tokens, decodes those tokens with the attached codec, and returns a waveform plus its valid length.

The public checkpoint provides five baked voices. The mapping is:

| Voice | Speaker index |
|---|---:|
| Aria | 0 |
| Jason | 1 |
| John | 2 |
| Leo | 3 |
| Sofia | 4 |

In [ ]:
SPEAKERS = {"Aria": 0, "Jason": 1, "John": 2, "Leo": 3, "Sofia": 4}

transcript = "Hello! Magpie generates codec tokens and turns them into speech."
language = "en"
speaker = "Sofia"

with torch.inference_mode():
    audio, audio_len = model.do_tts(
        transcript=transcript,
        language=language,
        apply_TN=True,
        use_cfg=True,
        speaker_index=SPEAKERS[speaker],
    )

num_samples = int(audio_len[0].item())
waveform = audio[0, :num_samples].float().cpu()

print(f"Generated {num_samples / model.output_sample_rate:.2f} seconds")
ipd.Audio(waveform.numpy(), rate=model.output_sample_rate)

In [ ]:
output_path = Path("magpie_tts_output.wav")
sf.write(output_path, waveform.numpy(), model.output_sample_rate)
print(f"Saved {output_path.resolve()}")

### Inspect the waveform

The generated output is already a waveform.

In [ ]:
time_axis = torch.arange(waveform.numel()) / model.output_sample_rate

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(time_axis, waveform, linewidth=0.6)
ax.set(title="Magpie-TTS waveform", xlabel="Time (seconds)", ylabel="Amplitude")
ax.grid(alpha=0.2)
plt.show()

## 5. Inspect the audio codec representation

The next cell sends the generated waveform back through the same codec. The encoder converts audio to discrete tokens, and the decoder reconstructs a waveform from those tokens.

This round trip is useful for understanding the representation that Magpie-TTS predicts. The standard `do_tts()` path already performs the decode step for you.

In [ ]:
codec_audio = audio[:, :num_samples].to(device)
codec_audio_len = torch.tensor([num_samples], dtype=torch.long, device=device)

with torch.inference_mode():
    audio_codes, audio_codes_len = codec.encode(
        audio=codec_audio,
        audio_len=codec_audio_len,
        sample_rate=model.output_sample_rate,
    )
    reconstructed_audio, reconstructed_audio_len = codec.decode(
        tokens=audio_codes,
        tokens_len=audio_codes_len,
    )

num_codec_frames = int(audio_codes_len[0].item())
num_reconstructed_samples = int(reconstructed_audio_len[0].item())
reconstructed_waveform = reconstructed_audio[0, :num_reconstructed_samples].float().cpu()

print(f"Audio-code tensor: {tuple(audio_codes.shape)}")
print(f"Valid codec frames: {num_codec_frames}")
print(f"Reconstructed duration: {num_reconstructed_samples / codec.output_sample_rate:.2f} seconds")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.imshow(
    audio_codes[0, :, :num_codec_frames].float().cpu(),
    origin="lower",
    aspect="auto",
    interpolation="nearest",
)
ax.set(title="Discrete audio-code tokens", xlabel="Codec frame", ylabel="Codebook")
plt.show()

In [ ]:
print("Original Magpie output")
ipd.display(ipd.Audio(waveform.numpy(), rate=model.output_sample_rate))

print("Codec reconstruction")
ipd.display(ipd.Audio(reconstructed_waveform.numpy(), rate=codec.output_sample_rate))

## 6. Languages and speakers

Select the tokenizer with the `language` argument and select a baked voice with `speaker_index`. The checkpoint supports the following languages and language codes:

    "Arabic-UAE": "ar-AE",
    "Arabic-Saudi Arabia": "ar-SA",
    "Arabic-MSA": "ar-MSA",
    "Chinese": "zh",
    "English": "en",
    "French": "fr",
    "German": "de",
    "Hindi": "hi",
    "Italian": "it",
    "Japanese": "ja",
    "Korean": "ko",
    "Brazilian Portuguese": "pt-BR",
    "Spanish": "es",
    "Vietnamese": "vi",

Use text written in the selected language. Text normalization can be enabled with `apply_TN=True` when the optional normalization dependencies support that language.

In [ ]:
examples = {
    "en": "Welcome to multilingual speech synthesis.",
    "es": "Bienvenidos a la sintesis de voz multilingue.",
    "de": "Willkommen bei der mehrsprachigen Sprachsynthese.",
    "fr": "Bienvenue dans la synthese vocale multilingue.",
}

selected_language = "fr"
selected_speaker = "Aria"

with torch.inference_mode():
    multilingual_audio, multilingual_len = model.do_tts(
        transcript=examples[selected_language],
        language=selected_language,
        apply_TN=False,
        use_cfg=True,
        speaker_index=SPEAKERS[selected_speaker],
    )

valid_len = int(multilingual_len[0].item())
ipd.Audio(
    multilingual_audio[0, :valid_len].float().cpu().numpy(),
    rate=model.output_sample_rate,
)

## 7. Generation controls

Magpie-TTS exposes two important robustness mechanisms:

- **Monotonic alignment** guides cross-attention from left to right, reducing skipped or repeated text.
- **Classifier-free guidance (CFG)** strengthens adherence to the selected conditioning. The convenience API enables CFG with `use_cfg=True`.

For advanced control over temperature, top-k sampling, context audio, attention priors, and batch evaluation, use `examples/tts/magpietts_inference.py`. The simple `do_tts()` API is intended for baked-speaker synthesis.

## 8. Measure real-time factor

Real-time factor (RTF) is generation time divided by generated audio duration. Values below 1 mean synthesis is faster than playback. The first call may include one-time CUDA setup, so warm up before measuring.

In [ ]:
benchmark_text = "Magpie produces robust speech from discrete audio tokens."

with torch.inference_mode():
    _ = model.do_tts(benchmark_text, language="en", speaker_index=0)

if torch.cuda.is_available():
    torch.cuda.synchronize()
start = time.perf_counter()
with torch.inference_mode():
    benchmark_audio, benchmark_len = model.do_tts(
        benchmark_text,
        language="en",
        use_cfg=True,
        speaker_index=0,
    )
if torch.cuda.is_available():
    torch.cuda.synchronize()
elapsed = time.perf_counter() - start

audio_seconds = int(benchmark_len[0].item()) / model.output_sample_rate
print(f"Generation time: {elapsed:.2f} seconds")
print(f"Audio duration: {audio_seconds:.2f} seconds")
print(f"Real-time factor: {elapsed / audio_seconds:.3f}")

## 9. Evaluation

A useful Magpie-TTS evaluation combines several perspectives:

- **Intelligibility:** transcribe generated audio with ASR and compute word or character error rate.
- **Speaker similarity:** compare speaker embeddings between generated and reference audio.
- **Codec fidelity:** compare original and codec-reconstructed waveforms or embeddings.
- **Alignment failures:** count repeated, skipped, truncated, or hallucinated words.
- **Naturalness:** use listening tests or a validated speech-quality estimator.
- **Efficiency:** report RTF, peak GPU memory, and generated audio duration.

The batch inference script can compute CER, speaker-similarity metrics, RTF, and other diagnostics from an evaluation manifest.

## 10. Troubleshooting

- **Out of memory:** use one GPU, close other models, and restart the kernel before loading Magpie-TTS again.
- **Model download fails:** verify Hugging Face access and retry `from_pretrained()`.
- **Unsupported language:** use one of the tokenizer language codes listed above.
- **Text normalization import error:** set `apply_TN=False` or install the TTS text-processing dependencies.
- **Custom voice conditioning:** use the Magpie inference runner with a compatible checkpoint and context-audio dataset; the public `do_tts()` example uses baked speakers.
- **Codec mismatch:** let the Magpie checkpoint load its configured codec. Do not pair it with an unrelated codec checkpoint because token codebooks and frame rates must match.

## 11. Additional resources

- [Magpie-TTS documentation](https://docs.nvidia.com/nemo/speech/nightly/tts/magpietts.html)
- [Magpie-TTS multilingual 357M model card](https://huggingface.co/nvidia/magpie_tts_multilingual_357m)
- [Magpie-TTS inference script](https://github.com/NVIDIA-NeMo/Speech/blob/main/examples/tts/magpietts_inference.py)
- [NeMo audio codec implementation](https://github.com/NVIDIA-NeMo/Speech/blob/main/nemo/collections/tts/models/audio_codec.py)